In [ ]:
#Initialization

In [ ]:
#Power factory connection with python
import os, sys
Dig_path=r"C:\Program Files\DIgSILENT\PowerFactory 2024\Python\3.12" # replace with ur actual path
sys.path.append(Dig_path)
os.environ['PATH']+=';'+Dig_path

import powerfactory as pf
import numpy as np
import pandas as pd

# 1. Initialize PowerFactory
app=pf.GetApplication()
if not app:
        raise Exception("Could not open PowerFactory application.")
print(app)

# 2. Activate your project
project_name = "sgphc"  # Replace with your actual project name
app.ActivateProject(project_name)

project = app.GetActiveProject()
ldf = app.GetFromStudyCase("ComLdf")
if project is None:
    raise Exception("Please activate your test sgphc Project.")
print(project)

In [ ]:
#Basecase analysis

In [ ]:
# -----------------------------
# Get NETWORK ELEMENTS
# -----------------------------
buses = app.GetCalcRelevantObjects("*.ElmTerm")
lines = app.GetCalcRelevantObjects("*.ElmLne")
trafos = app.GetCalcRelevantObjects("*.ElmTr2")
loads = app.GetCalcRelevantObjects("*.ElmLod")
ext_grids = app.GetCalcRelevantObjects("*.ElmXnet")
pv_units = app.GetCalcRelevantObjects("*.ElmGenstat")

# Susyem summary
import powerfactory as pf
import pandas as pd


def run_buswise_comparison():

    app = pf.GetApplication()
    app.ActivateProject("sgphc")

    ldf = app.GetFromStudyCase("ComLdf")

    terminals = app.GetCalcRelevantObjects("*.ElmTerm")
    lines = app.GetCalcRelevantObjects("*.ElmLne")
    pv_units = app.GetCalcRelevantObjects("*.ElmGenstat")

    # -----------------------------
    # STORE ORIGINAL PV SETTINGS
    # -----------------------------
    pv_original = {
        pv: {
            "pgini": pv.GetAttribute("pgini"),
            "qgini": pv.GetAttribute("qgini"),
            "outserv": pv.outserv
        }
        for pv in pv_units
    }

    # =============================
    # BASE CASE (NO PV)
    # =============================
    for pv in pv_units:
        pv.SetAttribute("pgini", 0.0)
        pv.SetAttribute("qgini", 0.0)

    ldf.Execute()

    base_voltage = {}
    bus_load = {}

    for term in terminals:
        bus = term.loc_name

        base_voltage[bus] = term.GetAttribute("m:u")

        p_load = sum(
            l.GetAttribute("plini")
            for l in term.GetConnectedElements()
            if l.GetClassName() == "ElmLod"
        )

        q_load = sum(
            l.GetAttribute("qlini")
            for l in term.GetConnectedElements()
            if l.GetClassName() == "ElmLod"
        )

        bus_load[bus] = (p_load, q_load)

    total_ploss_before = sum(l.GetAttribute("m:Ploss") for l in lines)
    total_qloss_before = sum(l.GetAttribute("m:Qloss") for l in lines)

    total_load_p = sum(v[0] for v in bus_load.values())
    total_load_q = sum(v[1] for v in bus_load.values())

    # =============================
    # PV CASE (50 kW EACH)
    # =============================
    injection_mw = 0.05  # 50 kW per PV

    for pv in pv_units:
        pv.SetAttribute("pgini", injection_mw)
        pv.SetAttribute("qgini", 0.0)

    ldf.Execute()

    post_voltage = {}

    # -----------------------------
    # CORRECT PV MAPPING PER BUS
    # -----------------------------
    bus_pv = {term.loc_name: (0.0, 0.0) for term in terminals}

    for pv in pv_units:

        if pv.outserv == 1:
            continue

        cub = pv.GetAttribute("bus1")
        if not cub:
            continue

        term = cub.GetAttribute("cterm")
        if not term:
            continue

        bus = term.loc_name

        p_kw = pv.GetAttribute("pgini") * 1000
        q_kvar = pv.GetAttribute("qgini") * 1000

        bus_pv[bus] = (
            bus_pv[bus][0] + p_kw,
            bus_pv[bus][1] + q_kvar
        )

    # collect post voltages
    for term in terminals:
        bus = term.loc_name
        post_voltage[bus] = term.GetAttribute("m:u")

    total_ploss_after = sum(l.GetAttribute("m:Ploss") for l in lines)
    total_qloss_after = sum(l.GetAttribute("m:Qloss") for l in lines)

    total_pv_p = sum(v[0] for v in bus_pv.values())
    total_pv_q = sum(v[1] for v in bus_pv.values())

    # =============================
    # EXTERNAL INFEED CALCULATION
    # =============================
    external_infeed_kw = total_load_p + total_ploss_after - total_pv_p

    # =============================
    # BUS-WISE TABLE
    # =============================
    data = []

    for bus in base_voltage.keys():

        v_base = base_voltage[bus]
        v_after = post_voltage[bus]
        dv = v_after - v_base
        dv_pct = (dv / v_base) * 100 if v_base else 0

        data.append({
            "Bus": bus,
            "V_base (pu)": v_base,
            "V_after (pu)": v_after,
            "ΔV (pu)": dv,
            "ΔV (%)": dv_pct,
            "Load P (kW)": bus_load[bus][0],
            "Load Q (kVAr)": bus_load[bus][1],
            "PV P (kW)": bus_pv[bus][0],
            "PV Q (kVAr)": bus_pv[bus][1],
        })

    df_bus = pd.DataFrame(data)

    # Natural sorting of buses
    df_bus["Bus_sort"] = df_bus["Bus"].str.extract(r"(\d+)").astype(float)
    df_bus = df_bus.sort_values(by="Bus_sort").drop(columns=["Bus_sort"])

    # =============================
    # CLEAN SYSTEM SUMMARY
    # =============================
    summary = {
        "Total Load P (kW)": total_load_p,
        "Total Load Q (kVAr)": total_load_q,

        "Total PV P (kW)": total_pv_p,
        "Total PV Q (kVAr)": total_pv_q,

        "Loss P Before (kW)": total_ploss_before,
        "Loss P After (kW)": total_ploss_after,
        "Δ Loss P (kW)": total_ploss_after - total_ploss_before,

        "Loss Q Before (kVAr)": total_qloss_before,
        "Loss Q After (kVAr)": total_qloss_after,
        "Δ Loss Q (kVAr)": total_qloss_after - total_qloss_before,

        "External Infeed (kW)": external_infeed_kw,

        "Power Balance Check (kW)": (
            external_infeed_kw + total_pv_p - total_ploss_after - total_load_p
        ),

        "No. of Buses": len(terminals),
        "No. of Lines": len(lines),
        "No. of PV Units": len(pv_units)
    }

    # =============================
    # PRINT RESULTS
    # =============================
    print("\n================ BUS-WISE RESULTS ================")
    print(df_bus.round(4).to_string(index=False))

    print("\n================ SYSTEM SUMMARY ================")
    print("-" * 65)

    for k, v in summary.items():
        if isinstance(v, float):
            print(f"{k:<40} : {v:>12.4f}")
        else:
            print(f"{k:<40} : {v:>12}")

    print("-" * 65)

    # =============================
    # POWER BALANCE CHECK
    # =============================
    tolerance = 1e-3
    balance_error = abs(
        external_infeed_kw + total_pv_p - total_ploss_after - total_load_p
    )

    if balance_error < tolerance:
        print("\n✔ Power balance satisfied")
    else:
        print(f"\n⚠ Power mismatch detected: {balance_error:.6f} kW")

    # =============================
    # RESTORE ORIGINAL PV SETTINGS
    # =============================
    for pv in pv_units:
        pv.SetAttribute("pgini", pv_original[pv]["pgini"])
        pv.SetAttribute("qgini", pv_original[pv]["qgini"])
        pv.outserv = pv_original[pv]["outserv"]

    ldf.Execute()

    return df_bus, summary


if __name__ == "__main__":
    run_buswise_comparison()

# base case restore 

import powerfactory as pf

def restore_base_case():

    app = pf.GetApplication()
    app.ActivateProject("sgphc")

    ldf = app.GetFromStudyCase("ComLdf")

    pv_units = app.GetCalcRelevantObjects("*.ElmGenstat")
    loads = app.GetCalcRelevantObjects("*.ElmLod")
    lines = app.GetCalcRelevantObjects("*.ElmLne")

    # ---------------------------
    # RESET PV → ZERO
    # ---------------------------
    for pv in pv_units:
        pv.SetAttribute("pgini", 0.0)
        pv.SetAttribute("qgini", 0.0)
        pv.outserv = 0  # ensure in service

    # ---------------------------
    # RUN LOAD FLOW
    # ---------------------------
    ldf.Execute()

    # ---------------------------
    # CHECK VALUES
    # ---------------------------
    total_load = sum(l.GetAttribute("plini") for l in loads)
    total_loss = sum(l.GetAttribute("m:Ploss") for l in lines)
    total_pv = sum(pv.GetAttribute("pgini") for pv in pv_units) * 1000

    print("\n===== BASE CASE RESTORED =====")
    print(f"Total Load (kW): {round(total_load,2)}")
    print(f"Total Loss (kW): {round(total_loss,2)}")
    print(f"Total PV (kW): {round(total_pv,2)}")

    return total_load, total_loss, total_pv


if __name__ == "__main__":
    restore_base_case()

In [ ]:
# DETERMINISTIC HOSTING CAPACITY ANALYSIS

In [ ]:
# ============================================================
# SORT BUSES
# ============================================================

def bus_sort(term):

    try:
        return int(term.loc_name)
    except:
        return 9999

terminals = sorted(terminals, key=bus_sort)

# ============================================================
# STORE ORIGINAL BASE CASE
# ============================================================

original_pv = {}

for pv in pv_units:

    original_pv[pv] = {
        "pgini": pv.GetAttribute("pgini"),
        "qgini": pv.GetAttribute("qgini"),
        "outserv": pv.outserv
    }

# ============================================================
# CREATE BUS LOAD MAP
# ============================================================

bus_load_map = {}

for term in terminals:

    p = 0.0
    q = 0.0

    for elem in term.GetConnectedElements():

        if elem.GetClassName() == "ElmLod":

            p += elem.GetAttribute("plini")
            q += elem.GetAttribute("qlini")

    bus_load_map[term.loc_name] = {
        "P_kW": p,
        "Q_kVAr": q
    }

# ============================================================
# CREATE PV MAP
# ============================================================

bus_pv_map = {}

for pv in pv_units:

    try:

        cub = pv.GetAttribute("bus1")

        if cub is None:
            continue

        term = cub.GetAttribute("cterm")

        if term is None:
            continue

        bus_pv_map[term.loc_name] = pv

    except:
        pass

# ============================================================
# RESET ALL PV
# ============================================================

def reset_all_pv():

    for pv in pv_units:

        pv.SetAttribute("pgini", 0.0)
        pv.SetAttribute("qgini", 0.0)

        pv.outserv = 0

# ============================================================
# RESTORE ORIGINAL BASE CASE
# ============================================================

def restore_original_base_case():

    for pv in pv_units:

        pv.SetAttribute(
            "pgini",
            original_pv[pv]["pgini"]
        )

        pv.SetAttribute(
            "qgini",
            original_pv[pv]["qgini"]
        )

        pv.outserv = original_pv[pv]["outserv"]

    ldf.Execute()

# ============================================================
# APPLY PV PENETRATION
# ============================================================

def apply_pv(alpha):

    total_pv_kw = 0.0

    for bus, pv in bus_pv_map.items():

        load_kw = bus_load_map[bus]["P_kW"]

        pv_kw = alpha * load_kw

        pv_mw = pv_kw / 1000

        pv.SetAttribute("pgini", pv_mw)

        pv.SetAttribute("qgini", 0.0)

        total_pv_kw += pv_kw

    return total_pv_kw

# ============================================================
# RUN LOAD FLOW
# ============================================================

def run_loadflow():

    err = ldf.Execute()

    return err

# ============================================================
# GET SYSTEM METRICS
# ============================================================

def get_system_metrics():

    voltages = []

    for term in terminals:

        voltages.append(term.GetAttribute("m:u"))

    vmin = min(voltages)
    vmax = max(voltages)

    total_loss = sum(
        line.GetAttribute("m:Ploss")
        for line in lines
    )

    total_load = sum(
        load.GetAttribute("plini")
        for load in loads
    )

    max_loading = max(
        line.GetAttribute("c:loading")
        for line in lines
    )

    voltage_violation = 0

    for v in voltages:

        if v > 1.05:
            voltage_violation += 1

    thermal_violation = 0

    for line in lines:

        if line.GetAttribute("c:loading") > 100:

            thermal_violation += 1

    # Reverse Power Flow
    rpf = 0

    for eg in ext_grids:

        try:

            pgrid = eg.GetAttribute("m:Psum:bus1")

            if pgrid < 0:

                rpf = 1

        except:
            pass

    return {
        "Vmin": vmin,
        "Vmax": vmax,
        "Loss_kW": total_loss,
        "Load_kW": total_load,
        "MaxLoading": max_loading,
        "VoltageViolation": voltage_violation,
        "ThermalViolation": thermal_violation,
        "RPF": rpf
    }

# ============================================================
# GET LINE LOSSES
# ============================================================

def get_line_losses():

    data = []

    for line in lines:

        data.append({
            "Line": line.loc_name,
            "P_loss_kW": line.GetAttribute("m:Ploss")
        })

    return pd.DataFrame(data)

# ============================================================
# GET LINE LOADING
# ============================================================

def get_line_loading():

    data = []

    for line in lines:

        data.append({
            "Line": line.loc_name,
            "Loading_percent": line.GetAttribute("c:loading")
        })

    return pd.DataFrame(data)

# ============================================================
# BASE CASE
# ============================================================

print("\n================================================")
print("RUNNING BASE CASE")
print("================================================")

reset_all_pv()

run_loadflow()

base_metrics = get_system_metrics()

base_line_losses = get_line_losses()

base_line_loading = get_line_loading()

print(f"\nBase Min Voltage : {base_metrics['Vmin']:.4f}")
print(f"Base Max Voltage : {base_metrics['Vmax']:.4f}")
print(f"Base Loss        : {base_metrics['Loss_kW']:.4f} kW")
print(f"Base Load        : {base_metrics['Load_kW']:.4f} kW")

# ============================================================
# DETERMINISTIC HC
# ============================================================

print("\n================================================")
print("RUNNING DETERMINISTIC HC")
print("================================================")

results = []

alpha = 0.0
step = 0.05
max_alpha = 5.0

rpf_started = False

rpf_alpha = None
rpf_total_pv = None

while alpha <= max_alpha:

    total_pv_kw = apply_pv(alpha)

    err = run_loadflow()

    if err != 0:

        print(f"Load flow failed at alpha={alpha}")

        break

    metrics = get_system_metrics()

    row = {
        "Alpha": alpha,
        "PV_kW": total_pv_kw,
        **metrics
    }

    results.append(row)

    print(
        f"Alpha={alpha:.2f} | "
        f"PV={total_pv_kw:.2f} kW | "
        f"Vmin={metrics['Vmin']:.4f} | "
        f"Vmax={metrics['Vmax']:.4f} | "
        f"Loss={metrics['Loss_kW']:.2f} kW | "
        f"MaxLoading={metrics['MaxLoading']:.2f}% | "
        f"RPF={metrics['RPF']}"
    )

    # RPF START
    if metrics["RPF"] == 1 and not rpf_started:

        rpf_started = True

        rpf_alpha = alpha
        rpf_total_pv = total_pv_kw

    # STOP CONDITIONS
    stop = False

    if metrics["Vmax"] > 1.05:

        print("\nVoltage violation reached")

        stop = True

    if metrics["ThermalViolation"] > 0:

        print("\nThermal violation reached")

        stop = True

    if stop:
        break

    alpha += step

# ============================================================
# RESULTS DATAFRAME
# ============================================================

df_results = pd.DataFrame(results)

# ============================================================
# HOSTING CAPACITY
# ============================================================

if len(df_results) >= 2:

    hc_row = df_results.iloc[-2]

else:

    hc_row = df_results.iloc[-1]

hc_alpha = hc_row["Alpha"]

# ============================================================
# RUN HC CASE AGAIN
# ============================================================

apply_pv(hc_alpha)

run_loadflow()

hc_metrics = get_system_metrics()

hc_line_losses = get_line_losses()

hc_line_loading = get_line_loading()

# ============================================================
# LINE LOSS COMPARISON
# ============================================================

loss_compare = pd.merge(
    base_line_losses,
    hc_line_losses,
    on="Line",
    suffixes=("_Base", "_HC")
)

loss_compare["Delta_Loss_kW"] = (
    loss_compare["P_loss_kW_HC"]
    - loss_compare["P_loss_kW_Base"]
)

# ============================================================
# LINE LOADING COMPARISON
# ============================================================

loading_compare = pd.merge(
    base_line_loading,
    hc_line_loading,
    on="Line",
    suffixes=("_Base", "_HC")
)

loading_compare["Delta_Loading_percent"] = (
    loading_compare["Loading_percent_HC"]
    - loading_compare["Loading_percent_Base"]
)

# ============================================================
# RPF BUS TABLE
# ============================================================

rpf_bus_data = []

if rpf_started:

    apply_pv(rpf_alpha)

    run_loadflow()

    for bus, pv in bus_pv_map.items():

        load_kw = bus_load_map[bus]["P_kW"]

        pv_kw = pv.GetAttribute("pgini") * 1000

        rpf_bus_data.append({
            "Bus": bus,
            "Connected_Load_kW": load_kw,
            "PV_Injection_kW": pv_kw
        })

df_rpf_bus = pd.DataFrame(rpf_bus_data)

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n================================================")
print("FINAL HOSTING CAPACITY RESULTS")
print("================================================")

print(f"\nHosting Capacity Alpha : {hc_alpha:.2f}")

print(f"Hosting Capacity PV    : {hc_row['PV_kW']:.2f} kW")

print(f"HC Min Voltage         : {hc_metrics['Vmin']:.4f}")

print(f"HC Max Voltage         : {hc_metrics['Vmax']:.4f}")

print(f"HC Loss                : {hc_metrics['Loss_kW']:.4f} kW")

print(f"HC Total Load          : {hc_metrics['Load_kW']:.4f} kW")

print(f"HC Max Loading         : {hc_metrics['MaxLoading']:.2f} %")

# ============================================================
# RPF RESULTS
# ============================================================

print("\n================================================")
print("REVERSE POWER FLOW")
print("================================================")

if rpf_started:

    print(f"\nRPF Starts at Alpha : {rpf_alpha:.2f}")

    print(f"RPF Starts at PV    : {rpf_total_pv:.2f} kW")

    print("\nBUS-WISE PV AT RPF")

    print(
        df_rpf_bus.round(2).to_string(index=False)
    )

else:

    print("\nNo Reverse Power Flow Detected")

# ============================================================
# LINE LOSS COMPARISON
# ============================================================

print("\n================================================")
print("LINE LOSS COMPARISON")
print("================================================")

print(
    loss_compare.round(4).to_string(index=False)
)

# ============================================================
# LINE LOADING COMPARISON
# ============================================================

print("\n================================================")
print("LINE LOADING COMPARISON")
print("================================================")

print(
    loading_compare.round(2).to_string(index=False)
)

# ============================================================
# EXPORT CSV FILES
# ============================================================

df_results.to_csv(
    "Deterministic_HC_Results.csv",
    index=False
)

loss_compare.to_csv(
    "Line_Loss_Comparison.csv",
    index=False
)

loading_compare.to_csv(
    "Line_Loading_Comparison.csv",
    index=False
)

df_rpf_bus.to_csv(
    "RPF_Bus_PV_Data.csv",
    index=False
)

print("\n================================================")
print("CSV FILES EXPORTED")
print("================================================")

# ============================================================
# RESTORE ORIGINAL BASE CASE
# ============================================================

print("\n================================================")
print("RESTORING ORIGINAL BASE CASE")
print("================================================")

restore_original_base_case()

# RUN AGAIN
run_loadflow()

restored_metrics = get_system_metrics()

print(f"\nRestored Min Voltage : {restored_metrics['Vmin']:.4f}")

print(f"Restored Max Voltage : {restored_metrics['Vmax']:.4f}")

print(f"Restored Loss        : {restored_metrics['Loss_kW']:.4f} kW")

print(f"Restored Load        : {restored_metrics['Load_kW']:.4f} kW")

print("\n================================================")
print("SCRIPT COMPLETED SUCCESSFULLY")  